# 中芯国际 A/H 股数据分析

本 Notebook 演示如何使用 **akshare** 获取中芯国际 A 股（688981）与港股（00981）近一年行情数据，并进行 K 线图、成交量图及 A/H 对比分析。

**数据来源：** akshare（前复权）  
**汇率来源：** 中国银行折算价（100 港元 → 人民币）

In [ ]:
# 如未安装依赖，取消下一行注释后运行
# !pip install akshare pandas matplotlib mplfinance -i https://pypi.tuna.tsinghua.edu.cn/simple

In [ ]:
from datetime import datetime, timedelta
from pathlib import Path

import akshare as ak
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import mplfinance as mpf
import pandas as pd

plt.rcParams["font.sans-serif"] = ["PingFang SC", "Heiti SC", "Arial Unicode MS", "SimHei"]
plt.rcParams["axes.unicode_minus"] = False

STOCK_NAME = "中芯国际"
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

end_dt = datetime.now()
start_dt = end_dt - timedelta(days=365)
start_date = start_dt.strftime("%Y%m%d")
end_date = end_dt.strftime("%Y%m%d")

print(f"分析区间: {start_dt.date()} ~ {end_dt.date()}")

## 1. 获取 A 股数据（688981）

使用 `stock_zh_a_daily` 接口，获取上交所科创板股票日线数据（前复权）。

In [ ]:
a_df = ak.stock_zh_a_daily(
    symbol="sh688981",
    start_date=start_date,
    end_date=end_date,
    adjust="qfq",
)
a_df = a_df.sort_values("date").reset_index(drop=True)
a_df["date"] = pd.to_datetime(a_df["date"])

print(f"A股共 {len(a_df)} 条记录")
a_df.tail()

## 2. 获取港股数据（00981）

使用 `stock_hk_daily` 接口，获取港交所日线数据（前复权）。

In [ ]:
hk_df = ak.stock_hk_daily(symbol="00981", adjust="qfq")
hk_df["date"] = pd.to_datetime(hk_df["date"])
hk_df = hk_df[hk_df["date"] >= start_dt].sort_values("date").reset_index(drop=True)

print(f"港股共 {len(hk_df)} 条记录")
hk_df.tail()

## 3. 获取港元/人民币汇率

用于计算 AH 溢价率：  
`AH溢价 = A股收盘价 ÷ (H股收盘价 × 汇率) − 1`

In [ ]:
fx_df = ak.currency_boc_sina(symbol="港币", start_date=start_date, end_date=end_date)
fx_df = fx_df.rename(columns={"日期": "date", "中行折算价": "rate"})
fx_df["date"] = pd.to_datetime(fx_df["date"])
fx_df["hkd_cny"] = pd.to_numeric(fx_df["rate"], errors="coerce") / 100  # 100 HKD = rate CNY
fx_df = fx_df[["date", "hkd_cny"]].dropna()

print(f"汇率共 {len(fx_df)} 条记录，最新 1 HKD = {fx_df['hkd_cny'].iloc[-1]:.4f} CNY")
fx_df.tail()

## 4. 绘制 A 股 K 线图 + 成交量

In [ ]:
def to_ohlcv(df: pd.DataFrame) -> pd.DataFrame:
    """转换为 mplfinance 需要的 OHLCV 格式。"""
    ohlcv = df.set_index("date")[["open", "high", "low", "close", "volume"]].copy()
    ohlcv.index.name = "Date"
    ohlcv.columns = ["Open", "High", "Low", "Close", "Volume"]
    return ohlcv


mc = mpf.make_marketcolors(up="#ef5350", down="#26a69a", inherit=True)
style = mpf.make_mpf_style(
    marketcolors=mc,
    gridstyle="--",
    gridcolor="#e0e0e0",
    facecolor="white",
    figcolor="white",
)

mpf.plot(
    to_ohlcv(a_df),
    type="candle",
    style=style,
    title=f"{STOCK_NAME} A股 688981 · 日K + 成交量",
    ylabel="价格 (元)",
    ylabel_lower="成交量",
    volume=True,
    figsize=(14, 7),
    datetime_format="%Y-%m-%d",
);

## 5. 绘制港股 K 线图 + 成交量

In [ ]:
mpf.plot(
    to_ohlcv(hk_df),
    type="candle",
    style=style,
    title=f"{STOCK_NAME} 港股 00981 · 日K + 成交量",
    ylabel="价格 (港元)",
    ylabel_lower="成交量",
    volume=True,
    figsize=(14, 7),
    datetime_format="%Y-%m-%d",
);

## 6. A/H 对比分析

在重叠交易日上对比：
- 归一化走势（起点 = 100）
- AH 溢价率
- 成交量对比

In [ ]:
merged = a_df.merge(hk_df, on="date", suffixes=("_a", "_hk"))
merged = merged.sort_values("date")
merged = pd.merge_asof(
    merged,
    fx_df.sort_values("date"),
    on="date",
    direction="backward",
).dropna(subset=["hkd_cny"])

merged["a_index"] = merged["close_a"] / merged["close_a"].iloc[0] * 100
merged["hk_index"] = merged["close_hk"] / merged["close_hk"].iloc[0] * 100
merged["hk_cny"] = merged["close_hk"] * merged["hkd_cny"]
merged["ah_premium"] = (merged["close_a"] / merged["hk_cny"] - 1) * 100

latest = merged.iloc[-1]
first = merged.iloc[0]
a_ret = (latest["close_a"] / first["close_a"] - 1) * 100
hk_ret = (latest["close_hk"] / first["close_hk"] - 1) * 100

print(f"重叠交易日: {len(merged)} 天 ({merged['date'].iloc[0].date()} ~ {merged['date'].iloc[-1].date()})")
print(f"A股区间涨跌: {a_ret:+.2f}%")
print(f"港股区间涨跌: {hk_ret:+.2f}%")
print(f"A-H 涨跌差: {a_ret - hk_ret:+.2f}%")
print(f"最新 AH 溢价率: {latest['ah_premium']:+.2f}%")
print(f"平均 AH 溢价率: {merged['ah_premium'].mean():+.2f}%")

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)
fig.suptitle(f"{STOCK_NAME} A/H 股对比分析", fontsize=16, fontweight="bold")

# 归一化走势
ax1 = axes[0]
ax1.plot(merged["date"], merged["a_index"], color="#5b8def", linewidth=1.8, label="A股 688981")
ax1.plot(merged["date"], merged["hk_index"], color="#f5a623", linewidth=1.8, label="港股 00981")
ax1.set_ylabel("归一化指数 (起点=100)")
ax1.legend(loc="upper left")
ax1.grid(True, linestyle="--", alpha=0.4)
ax1.set_title("归一化走势对比")

# AH 溢价率
ax2 = axes[1]
ax2.plot(merged["date"], merged["ah_premium"], color="#b388ff", linewidth=1.8)
ax2.axhline(0, color="#888", linestyle="--", linewidth=1)
ax2.fill_between(merged["date"], merged["ah_premium"], 0, alpha=0.15, color="#b388ff")
ax2.set_ylabel("AH 溢价率 (%)")
ax2.grid(True, linestyle="--", alpha=0.4)
ax2.set_title("AH 溢价率 = A股收盘价 ÷ (H股收盘价 × 汇率) − 1")

# 成交量对比（双 Y 轴）
ax3 = axes[2]
ax3.bar(merged["date"], merged["volume_a"], width=1.5, alpha=0.6, color="#5b8def", label="A股成交量")
ax3.set_ylabel("A股成交量", color="#5b8def")
ax3.tick_params(axis="y", labelcolor="#5b8def")
ax3.grid(True, linestyle="--", alpha=0.4)

ax3b = ax3.twinx()
ax3b.plot(merged["date"], merged["volume_hk"], color="#f5a623", linewidth=1.5, label="港股成交量")
ax3b.set_ylabel("港股成交量", color="#f5a623")
ax3b.tick_params(axis="y", labelcolor="#f5a623")
ax3.set_title("成交量对比")

ax3.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 7. 保存数据到本地

In [ ]:
a_df.assign(date=a_df["date"].dt.strftime("%Y-%m-%d")).to_csv(
    DATA_DIR / "smic_688981_a.csv", index=False, encoding="utf-8-sig"
)
hk_df.assign(date=hk_df["date"].dt.strftime("%Y-%m-%d")).to_csv(
    DATA_DIR / "smic_00981_hk.csv", index=False, encoding="utf-8-sig"
)
merged.to_csv(DATA_DIR / "smic_compare_merged.csv", index=False, encoding="utf-8-sig")

print("已保存:")
print(f"  {DATA_DIR / 'smic_688981_a.csv'}")
print(f"  {DATA_DIR / 'smic_00981_hk.csv'}")
print(f"  {DATA_DIR / 'smic_compare_merged.csv'}")